# Ìmọ̀lára · `e4_afriberta_large_final`

E4: AfriBERTa large (126M), pre-trained from scratch on <1 GB of 11 African languages incl. Yoruba

Runs `python -m src.train_transformer --config configs/e4_afriberta_large.yaml` for seeds 42, 43, 44 with `--final` on a Kaggle T4 GPU.

* **Code:** [Hassan-Adelani-Luqman/imolara-yoruba-sentiment](https://github.com/Hassan-Adelani-Luqman/imolara-yoruba-sentiment) at commit [`4844906`](https://github.com/Hassan-Adelani-Luqman/imolara-yoruba-sentiment/tree/4844906ff8b15563af54dcbabfae2ad2fc19ff39), cloned below; nothing is edited inside this notebook.
* **Data:** AfriSenti Yoruba (raw TSVs, pinned commit), downloaded by `src/data.py`.
* **Outputs:** `/kaggle/working/outputs/seed*/` (metrics, dev predictions, run info), downloaded to `results/kaggle/e4_afriberta_large_final/` in the repo.
* **Reproduce:** *Copy & Edit*, set Accelerator = GPU T4 and Internet = on, then *Run all*.

Generated by `scripts/kaggle_run.py --notebook`.

## 1 · Setup: clone the repo at the pinned commit, install extras, check the GPU

In [1]:
import glob, json, os, subprocess, sys

REPO, COMMIT = "https://github.com/Hassan-Adelani-Luqman/imolara-yoruba-sentiment.git", "4844906ff8b15563af54dcbabfae2ad2fc19ff39"
CODE, OUT = "/tmp/imolara", "/kaggle/working/outputs"
CONFIG, SEEDS = "configs/e4_afriberta_large.yaml", [42, 43, 44]

subprocess.run(["git", "clone", "-q", REPO, CODE], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "-q", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements-kaggle.txt"], check=True)
# Kaggle ships torchao 0.10, which peft >= 0.20 rejects at import time; we don't use it.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)

import torch
assert torch.cuda.is_available(), "No GPU: set Accelerator = GPU T4"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

os.makedirs(OUT, exist_ok=True)
with open(f"{OUT}/env.txt", "w") as f:            # exact package versions, for the report
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=f, check=True)

env = {**os.environ, "IMOLARA_GIT_COMMIT": COMMIT[:7], "PYTHONUNBUFFERED": "1"}
emb = glob.glob("/kaggle/input/**/w2v_yo_300.kv", recursive=True)
if emb:                                            # attached imolara-embeddings dataset (RNN runs)
    env["IMOLARA_EMB_DIR"] = os.path.dirname(emb[0])
    print("embeddings:", env["IMOLARA_EMB_DIR"])

GPU: Tesla T4 | torch 2.11.0+cu128


## 2 · Experiment configuration

In [2]:
print(open(f"{CODE}/{CONFIG}").read())

# E4: AfriBERTa large (126M), pre-trained from scratch on <1 GB of 11 African languages incl. Yoruba
model_name: castorini/afriberta_large
learning_rate: 3.0e-5
batch_size: 32
epochs: 10
early_stopping_patience: 3
eval_diacritics: [original, no_tones, no_diacritics]



## 3 · Training: one process per seed, one GPU per process

Kaggle's T4 machine has two GPUs. Each seed trains in its own process pinned to one GPU (`CUDA_VISIBLE_DEVICES`), so two seeds run at once and `batch_size` in the config is the real batch (no DataParallel). Log lines are prefixed with their seed. The best epoch is chosen on *clean* dev macro-F1 (dev tweets that do not duplicate a training tweet).

In [3]:
import queue, threading

N_GPU = max(1, torch.cuda.device_count())
print(f"{N_GPU} GPU(s) -> up to {N_GPU} seeds in parallel", flush=True)
NOISE = ("Warning", "warn(", "it/s]", "Loading weights")

def run(seed, gpu):
    cmd = [sys.executable, "-m", "src.train_transformer", "--config", CONFIG, "--seed", str(seed),
           "--output_dir", f"{OUT}/seed{seed}", *['--final']]
    proc = subprocess.Popen(cmd, cwd=CODE, env={**env, "CUDA_VISIBLE_DEVICES": str(gpu)},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if not any(n in line for n in NOISE):
            print(f"[seed {seed} | gpu {gpu}] {line}", end="", flush=True)
    return proc.wait()

def run_all(seeds):
    todo, codes = queue.Queue(), {}
    for s in seeds:
        todo.put(s)
    def worker(gpu):
        while True:
            try:
                seed = todo.get_nowait()
            except queue.Empty:
                return
            codes[seed] = run(seed, gpu)
    threads = [threading.Thread(target=worker, args=(g,)) for g in range(N_GPU)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    failed = [s for s, rc in codes.items() if rc != 0]
    assert not failed, f"seeds failed: {failed}"

run_all(SEEDS)

2 GPU(s) -> up to 2 seeds in parallel
[seed 43 | gpu 1] 
[seed 42 | gpu 0] 
[seed 43 | gpu 1] [transformers] XLMRobertaForSequenceClassification LOAD REPORT from: castorini/afriberta_large
[seed 42 | gpu 0] [transformers] XLMRobertaForSequenceClassification LOAD REPORT from: castorini/afriberta_large
[seed 43 | gpu 1] Key                        | Status     | 
[seed 43 | gpu 1] ---------------------------+------------+-
[seed 42 | gpu 0] Key                        | Status     | 
[seed 42 | gpu 0] ---------------------------+------------+-
[seed 43 | gpu 1] lm_head.layer_norm.weight  | UNEXPECTED | 
[seed 42 | gpu 0] lm_head.decoder.bias       | UNEXPECTED | 
[seed 43 | gpu 1] lm_head.dense.bias         | UNEXPECTED | 
[seed 43 | gpu 1] lm_head.layer_norm.bias    | UNEXPECTED | 
[seed 42 | gpu 0] lm_head.layer_norm.weight  | UNEXPECTED | 
[seed 43 | gpu 1] lm_head.bias               | UNEXPECTED | 
[seed 42 | gpu 0] lm_head.decoder.weight     | UNEXPECTED | 
[seed 43 | gpu 1] lm_head.d

## 4 · Results across seeds

Macro-F1 is the primary metric; weighted-F1 is the official SemEval-2023 metric. `clean` excludes tweets that duplicate a training tweet; `no_tones` / `no_diacritics` score the same tweets with tone marks / all diacritics removed.

In [4]:
import pandas as pd

rows = [r for s in SEEDS for r in json.load(open(f"{OUT}/seed{s}/metrics.json"))]
df = pd.DataFrame(rows)
summary = (df.groupby(["split", "subset", "eval_diacritics"])[["macro_f1", "weighted_f1", "accuracy"]]
             .agg(["mean", "std"]).round(4))
summary

macro_f1         weighted_f1         accuracy  \
                                 mean     std        mean     std     mean   
split subset eval_diacritics                                                 
dev   all    no_diacritics     0.6799  0.0061      0.6998  0.0062   0.7027   
             no_tones          0.6973  0.0012      0.7185  0.0013   0.7193   
             original          0.7602  0.0033      0.7757  0.0029   0.7757   
      clean  no_diacritics     0.6582  0.0068      0.6787  0.0063   0.6815   
             no_tones          0.6728  0.0018      0.6945  0.0021   0.6951   
             original          0.7320  0.0037      0.7485  0.0033   0.7483   
test  all    no_diacritics     0.6778  0.0109      0.6984  0.0107   0.7000   
             no_tones          0.7102  0.0068      0.7291  0.0055   0.7289   
             original          0.7505  0.0042      0.7666  0.0034   0.7662   
      clean  no_diacritics     0.6624  0.0105      0.6827  0.0101   0.6841   
             no_tones          0.6939  0.0054      0.7128  0.0043   0.7125   
             original          0.7321  0.0047      0.7488  0.0039   0.7484   

                                      
                                 std  
split subset eval_diacritics          
dev   all    no_diacritics    0.0052  
             no_tones         0.0020  
             original         0.0028  
      clean  no_diacritics    0.0061  
             no_tones         0.0028  
             original         0.0031  
test  all    no_diacritics    0.0086  
             no_tones         0.0053  
             original         0.0033  
      clean  no_diacritics    0.0080  
             no_tones         0.0043  
             original         0.0038

In [5]:
per_seed = df[(df.subset == "clean") & (df.eval_diacritics == "original")][
    ["split", "seed", "macro_f1", "weighted_f1", "f1_negative", "f1_neutral", "f1_positive"]]
per_seed

,split,seed,macro_f1,weighted_f1,f1_negative,f1_neutral,f1_positive
1,dev,42,0.7309,0.7477,0.6474,0.7411,0.8041
7,test,42,0.7374,0.7533,0.6546,0.7496,0.8081
13,dev,43,0.7290,0.7457,0.6449,0.7423,0.7997
19,test,43,0.7303,0.7467,0.6447,0.7431,0.8031
25,dev,44,0.7361,0.7521,0.6573,0.7439,0.8071
31,test,44,0.7285,0.7465,0.6353,0.7410,0.8093
